# 03 — Experiments: RMSE decomposition + CatBoost log/MAE vs log/RMSE
Reproduces every post-baseline number quoted in `report/report.pdf` section 7 on the identical 3 temporal folds and features. Run from the `notebooks/` directory. Training takes several minutes.

In [1]:
import sys; sys.path.insert(0, '..')
from pathlib import Path
import numpy as np, pandas as pd
from src.cleaning import clean_frame
from src.features import build_features, feature_matrix
from src.model import fit_predict_log_l1
DATA = Path('../data')
FOLDS = [('2025-05-01', '2025-07-01'), ('2025-07-01', '2025-09-01'), ('2025-09-01', '2025-11-01')]
df = pd.read_csv(DATA / 'train_test.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

def fold_data(cut, end):
    cut, end = pd.Timestamp(cut), pd.Timestamp(end)
    dtr = df[df['date'] < cut].copy(); dte = df[(df['date'] >= cut) & (df['date'] < end)].copy()
    med_w = float(dtr['weight'].abs().median())
    dm = dtr.groupby(dtr['date'].astype(str))['market_index'].median()
    gm, qm = float(dtr['market_index'].median()), float(dtr['quote_signal'].median())
    dtr_c, dte_c = clean_frame(dtr, med_w, dm, gm), clean_frame(dte, med_w, dm, gm)
    full = pd.concat([dtr_c, dte_c])
    fmap = full.groupby(pd.to_datetime(full['date']).dt.strftime('%Y-%m-%d'))['market_index'].median()
    Xtr = feature_matrix(build_features(dtr_c, qm, fmap)).to_numpy()
    Xte = feature_matrix(build_features(dte_c, qm, fmap)).to_numpy()
    return Xtr, Xte, dtr['posted_rate'].to_numpy(float), dte['posted_rate'].to_numpy(float), dte

def show(y, p, tag):
    p = np.clip(p, 1.0, None); ape = np.abs(p - y) / y
    print(f'{tag}: MAE={np.abs(p - y).mean():.1f} RMSE={np.sqrt(((p - y) ** 2).mean()):.1f} MAPE={ape.mean() * 100:.2f}%', flush=True)

## 1. RMSE decomposition (Fold3 LGBM-L1): full vs trimmed

In [2]:
Xtr, Xte, ytr, yte, dte = fold_data(*FOLDS[2])
p = np.clip(fit_predict_log_l1(Xtr, np.log1p(ytr), Xte, seed=0), 1.0, None)
show(yte, p, 'ALL rows          ')
ae = np.abs(p - yte); order = np.argsort(ae)
for frac, tag in [(0.985, 'drop worst 1.5% '), (0.95, 'drop worst 5%   ')]:
    idx = order[:int(len(order) * frac)]
    show(yte[idx], p[idx], tag)
print('\nworst 8 (true vs pred @ distance):')
for i in order[-8:][::-1]:
    print(f"  true={yte[i]:.0f} pred={p[i]:.0f} dist={dte.iloc[i]['distance']:.0f}")

C:\Users\Qais\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ALL rows          : MAE=106.5 RMSE=633.9 MAPE=4.66%
drop worst 1.5% : MAE=50.5 RMSE=74.0 MAPE=2.32%
drop worst 5%   : MAE=43.4 RMSE=57.5 MAPE=2.07%

worst 8 (true vs pred @ distance):
  true=20132 pred=3860 dist=1760
  true=20362 pred=4346 dist=2283
  true=17514 pred=3581 dist=1893
  true=17893 pred=4112 dist=2023
  true=19110 pred=5976 dist=2979
  true=14949 pred=3208 dist=1640
  true=14784 pred=3584 dist=1684
  true=13504 pred=2917 dist=1460


## 2. CatBoost on log-target: MAE loss vs RMSE loss (all 3 folds)

In [3]:
from catboost import CatBoostRegressor
for loss in ['MAE', 'RMSE']:
    maes = []
    for cut, end in FOLDS:
        Xtr, Xte, ytr, yte, _ = fold_data(cut, end)
        n = len(Xtr); k = max(500, n // 10); tr_idx, ev_idx = np.arange(n - k), np.arange(n - k, n)
        m = CatBoostRegressor(iterations=1500, learning_rate=0.04, depth=7, l2_leaf_reg=5.0,
                                loss_function=loss, random_seed=0, thread_count=-1, verbose=False)
        m.fit(Xtr[tr_idx], np.log1p(ytr[tr_idx]), eval_set=(Xtr[ev_idx], np.log1p(ytr[ev_idx])),
              early_stopping_rounds=150, verbose=False)
        pf = np.clip(np.expm1(m.predict(Xte)), 1.0, None)
        a = float(np.abs(pf - yte).mean()); maes.append(a)
        print(f'CatBoost-log{loss} fold {cut}: MAE={a:.1f} (best_iter={m.best_iteration_})', flush=True)
    print(f'>>> CatBoost-log{loss} MEAN MAE={np.mean(maes):.1f}\n', flush=True)
print('reference: lgbm-logL1 folds 119.4 / 122.0 / 106.5, mean 116.0')

CatBoost-logMAE fold 2025-05-01: MAE=138.1 (best_iter=1147)
CatBoost-logMAE fold 2025-07-01: MAE=121.8 (best_iter=1019)
CatBoost-logMAE fold 2025-09-01: MAE=118.8 (best_iter=1028)
>>> CatBoost-logMAE MEAN MAE=126.2

CatBoost-logRMSE fold 2025-05-01: MAE=128.9 (best_iter=382)
CatBoost-logRMSE fold 2025-07-01: MAE=100.5 (best_iter=301)
CatBoost-logRMSE fold 2025-09-01: MAE=110.3 (best_iter=184)
>>> CatBoost-logRMSE MEAN MAE=113.2

reference: lgbm-logL1 folds 119.4 / 122.0 / 106.5, mean 116.0
